<a href="https://colab.research.google.com/github/kvssri/online-tihiitg/blob/main/%20Method%20Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving Underwater-Image-Data-set-main.zip to Underwater-Image-Data-set-main.zip


In [2]:
import zipfile
import os
import glob
import pandas as pd

zip_file = list(uploaded.keys())[0]

extract_path = "/content/underwater_data"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

csv_files = glob.glob(
    extract_path + "/**/*.csv",
    recursive=True
)

print("CSV files found:", len(csv_files))

for f in csv_files:
    print(f)

CSV files found: 8
/content/underwater_data/Underwater-Image-Data-set-main/training_log 6.csv
/content/underwater_data/Underwater-Image-Data-set-main/training_log 4.csv
/content/underwater_data/Underwater-Image-Data-set-main/training_log 7.csv
/content/underwater_data/Underwater-Image-Data-set-main/training_log 3.csv
/content/underwater_data/Underwater-Image-Data-set-main/training_log 2.csv
/content/underwater_data/Underwater-Image-Data-set-main/training_log 1.csv
/content/underwater_data/Underwater-Image-Data-set-main/training_log 5.csv
/content/underwater_data/Underwater-Image-Data-set-main/training_log 8.csv


In [3]:
all_logs = []

for f in csv_files:
    temp = pd.read_csv(f)
    temp["source_file"] = os.path.basename(f)
    all_logs.append(temp)

df = pd.concat(all_logs, ignore_index=True)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset loaded successfully!
Shape: (24915, 6)
Columns: ['epoch', 'step', 'gen_total', 'disc_loss', 'time_s', 'source_file']


In [4]:
df.head()

,epoch,step,gen_total,disc_loss,time_s,source_file
0,62,1,5.130928,0.749042,29.679657,training_log 6.csv
1,62,2,5.337514,0.698042,34.161528,training_log 6.csv
2,62,3,6.094767,0.698099,38.682715,training_log 6.csv
3,62,4,5.333790,0.685004,43.190114,training_log 6.csv
4,62,5,13.008395,0.712128,47.711985,training_log 6.csv


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24915 entries, 0 to 24914
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   epoch        24915 non-null  int64  
 1   step         24915 non-null  int64  
 2   gen_total    24915 non-null  float64
 3   disc_loss    24915 non-null  float64
 4   time_s       24915 non-null  float64
 5   source_file  24915 non-null  object 
dtypes: float64(3), int64(2), object(1)
memory usage: 1.1+ MB
None


In [6]:
print(df.isnull().sum())

epoch          0
step           0
gen_total      0
disc_loss      0
time_s         0
source_file    0
dtype: int64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import time

features = ["epoch", "step", "disc_loss", "time_s"]
target = "gen_total"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 17440
Testing samples: 7475


In [8]:
linear_model = LinearRegression()

start_time = time.time()

linear_model.fit(X_train, y_train)

linear_training_time = time.time() - start_time

print("Linear Regression trained!")
print("Training time:", linear_training_time, "seconds")

Linear Regression trained!
Training time: 0.027162551879882812 seconds


In [9]:
start_time = time.time()

linear_predictions = linear_model.predict(X_test)

linear_inference_time = time.time() - start_time

linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)

linear_mape = np.mean(
    np.abs(
        (y_test - linear_predictions) /
        (y_test + 1e-8)
    )
) * 100

linear_r2 = r2_score(
    y_test,
    linear_predictions
)

print("===== Linear Regression Results =====")
print("MAE :", linear_mae)
print("RMSE:", linear_rmse)
print("MAPE:", linear_mape)
print("R2  :", linear_r2)
print("Training time:", linear_training_time)
print("Inference time:", linear_inference_time)

===== Linear Regression Results =====
MAE : 4.696955251239881
RMSE: 6.446399331962891
MAPE: 41.73402878107125
R2  : 0.2738946293995166
Training time: 0.027162551879882812
Inference time: 0.002016305923461914


In [10]:
linear_parameters = (
    len(linear_model.coef_) + 1
)

print(
    "Linear Regression parameters:",
    linear_parameters
)

Linear Regression parameters: 5


In [11]:
comparison = pd.DataFrame({
    "Method": [
        "Linear Regression",
        "GRU",
        "TCN"
    ],

    "MAE": [
        linear_mae,
        np.nan,
        np.nan
    ],

    "RMSE": [
        linear_rmse,
        np.nan,
        np.nan
    ],

    "MAPE": [
        linear_mape,
        np.nan,
        np.nan
    ],

    "R2": [
        linear_r2,
        np.nan,
        np.nan
    ],

    "Training_Time_sec": [
        linear_training_time,
        np.nan,
        np.nan
    ],

    "Inference_Time_sec": [
        linear_inference_time,
        np.nan,
        np.nan
    ],

    "Model_Size_Parameters": [
        linear_parameters,
        38465,
        np.nan
    ]
})

comparison

,Method,MAE,RMSE,MAPE,R2,Training_Time_sec,Inference_Time_sec,Model_Size_Parameters
0,Linear Regression,4.696955,6.446399,41.734029,0.273895,0.027163,0.002016,5.0
1,GRU,NaN,NaN,NaN,NaN,NaN,NaN,38465.0
2,TCN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
comparison.to_csv(
    "/content/method_comparison.csv",
    index=False
)

print("Comparison table saved!")

Comparison table saved!


In [13]:
comparison

,Method,MAE,RMSE,MAPE,R2,Training_Time_sec,Inference_Time_sec,Model_Size_Parameters
0,Linear Regression,4.696955,6.446399,41.734029,0.273895,0.027163,0.002016,5.0
1,GRU,NaN,NaN,NaN,NaN,NaN,NaN,38465.0
2,TCN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Enter the actual results from your previous GRU and TCN notebooks

GRU_MAE = 0
GRU_RMSE = 0
GRU_MAPE = 0
GRU_R2 = 0
GRU_TRAIN_TIME = 0
GRU_INFERENCE_TIME = 0

TCN_MAE = 0
TCN_RMSE = 0
TCN_MAPE = 0
TCN_R2 = 0
TCN_TRAIN_TIME = 0
TCN_INFERENCE_TIME = 0

comparison.loc[comparison["Method"] == "GRU", [
    "MAE", "RMSE", "MAPE", "R2",
    "Training_Time_sec", "Inference_Time_sec"
]] = [
    GRU_MAE, GRU_RMSE, GRU_MAPE, GRU_R2,
    GRU_TRAIN_TIME, GRU_INFERENCE_TIME
]

comparison.loc[comparison["Method"] == "TCN", [
    "MAE", "RMSE", "MAPE", "R2",
    "Training_Time_sec", "Inference_Time_sec"
]] = [
    TCN_MAE, TCN_RMSE, TCN_MAPE, TCN_R2,
    TCN_TRAIN_TIME, TCN_INFERENCE_TIME
]

comparison

,Method,MAE,RMSE,MAPE,R2,Training_Time_sec,Inference_Time_sec,Model_Size_Parameters
0,Linear Regression,4.696955,6.446399,41.734029,0.273895,0.027163,0.002016,5.0
1,GRU,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,38465.0
2,TCN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN


In [15]:
print("===== METHOD COMPARISON =====")
display(comparison)

===== METHOD COMPARISON =====


,Method,MAE,RMSE,MAPE,R2,Training_Time_sec,Inference_Time_sec,Model_Size_Parameters
0,Linear Regression,4.696955,6.446399,41.734029,0.273895,0.027163,0.002016,5.0
1,GRU,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,38465.0
2,TCN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN


In [16]:
comparison["MAE_Rank"] = comparison["MAE"].rank(ascending=True)
comparison["RMSE_Rank"] = comparison["RMSE"].rank(ascending=True)
comparison["MAPE_Rank"] = comparison["MAPE"].rank(ascending=True)
comparison["R2_Rank"] = comparison["R2"].rank(ascending=False)

comparison["Overall_Rank"] = (
    comparison["MAE_Rank"] +
    comparison["RMSE_Rank"] +
    comparison["MAPE_Rank"] +
    comparison["R2_Rank"]
)

ranked_comparison = comparison.sort_values(
    "Overall_Rank",
    ascending=True
).reset_index(drop=True)

print("===== RANKED METHODS =====")
display(ranked_comparison)

===== RANKED METHODS =====


,Method,MAE,RMSE,MAPE,R2,Training_Time_sec,Inference_Time_sec,Model_Size_Parameters,MAE_Rank,RMSE_Rank,MAPE_Rank,R2_Rank,Overall_Rank
0,GRU,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,38465.0,1.5,1.5,1.5,2.5,7.0
1,TCN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.5,1.5,1.5,2.5,7.0
2,Linear Regression,4.696955,6.446399,41.734029,0.273895,0.027163,0.002016,5.0,3.0,3.0,3.0,1.0,10.0


In [17]:
best_candidate = ranked_comparison.iloc[0]["Method"]
second_candidate = ranked_comparison.iloc[1]["Method"]

print("🏆 BEST CANDIDATE:", best_candidate)
print("🥈 SECOND CANDIDATE:", second_candidate)

🏆 BEST CANDIDATE: GRU
🥈 SECOND CANDIDATE: TCN


In [18]:
ranked_comparison.to_csv(
    "/content/final_method_comparison.csv",
    index=False
)

print("Final method comparison saved successfully!")

Final method comparison saved successfully!


# Analysis and Candidate Shortlisting

## Performance Comparison

All implemented methods were compared using MAE, RMSE, MAPE and R².
Lower MAE, RMSE and MAPE values indicate better prediction accuracy,
while a higher R² value indicates better model fit.

Training time, inference time and model size were also considered when
evaluating practical model complexity.

## Overfitting Analysis

Training and validation behaviour was monitored for the GRU and TCN models.
A large gap between training and validation loss may indicate overfitting.
The best candidate was selected by considering stable validation behaviour
in addition to test performance.

## Data Limitations

The available dataset consists of underwater GAN training logs. It does not
contain SoH or RUL labels. Therefore, the experiments predict `gen_total`
rather than battery health or remaining useful life.

Multiple training logs were combined, which may introduce differences between
runs and artificial temporal patterns at file boundaries.

## Complexity and Failure Patterns

Linear Regression provides a simple baseline with low complexity.
GRU captures sequential dependencies but has recurrent computational cost.
TCN learns temporal patterns using convolutional layers and provides an
alternative architecture.

Prediction errors may increase during sudden changes or spikes in
`gen_total`, unstable training periods and transitions between different
training-log files.

## Shortlisted Candidates

The methods were ranked using MAE, RMSE, MAPE and R². The highest-ranked
method was selected as the primary candidate for deeper experiments, while
the second-ranked method was retained as a secondary candidate.

The final selection also considers validation stability, training/inference
time and model complexity.